<a href="https://colab.research.google.com/github/swopnimghimire-123123/AI_Fellowship/blob/main/LSTMs_for_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTMs for Text Classification

**Dataset:** AG_NEWS (News topic classification: World, Sports, Business, Sci/Tech)

**Instructions:** Complete the simple `# TODO` sections marked with `None`. Run all cells top-to-bottom to train and evaluate your model.

In [ ]:
# Run this cell to install dependencies and load the dataset
!pip install -q datasets

In [ ]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Step 1: Vocabulary & Preprocessing
Neural networks need numbers, not raw text. We will build a vocabulary to map words to integers.

In [1]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
def tokenizer(text):
    return re.findall(r"[a-z0-9]+", text.lower())

ag_news = load_dataset('fancyzhx/ag_news')
train_data = ag_news['train']
test_data = ag_news['test']

def yield_tokens(data_iter):
    for example in data_iter:
        yield tokenizer(example['text'])

from collections import Counter

counter = Counter()
for tokens in yield_tokens(train_data):
    counter.update(tokens)

itos = ['<unk>', '<pad>'] + list(counter.keys())
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX = stoi['<unk>']
PAD_IDX = stoi['<pad>']

def numericalize(text):
    return [stoi.get(tok, UNK_IDX) for tok in tokenizer(text)]

print(f"Vocabulary size: {len(itos):,}")

def collate_batch(batch):
    label_list, text_list = [], []
    for example in batch:
        label_list.append(example['label'])  # ag_news labels are already 0-indexed (0-3)
        processed_text = torch.tensor(numericalize(example['text']), dtype=torch.int64)
        text_list.append(processed_text)

    # TODO: Use nn.utils.rnn.pad_sequence to pad text_list so all sentences in the batch are the same length.
    # Hint: set batch_first=True and padding_value=PAD_IDX
    padded_texts = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(label_list, dtype=torch.int64)

    return padded_texts, labels

# Using a small subset of data for fast training
train_list = list(train_data)[:5000]
test_list  = list(test_data)[:1000]

tr_ld = DataLoader(train_list, batch_size=32, shuffle=True, collate_fn=collate_batch)
te_ld = DataLoader(test_list, batch_size=32, shuffle=False, collate_fn=collate_batch)

Using device: cuda


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Vocabulary size: 65,017


### Step 2: Build the LSTM Model
Construct a simple LSTM text classifier.

In [2]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # TODO: Define a PyTorch nn.LSTM layer (set batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # TODO: Define a fully connected layer mapping hidden_dim to num_classes
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text):
        embedded = self.embedding(text)

        # TODO: Pass the embedded text through the LSTM
        # The LSTM returns two things: output and (hidden_state, cell_state)
        output, (hidden, cell) = self.lstm(embedded)

        # We only care about the final hidden state of the last layer for classification
        final_hidden = hidden[-1]

        # TODO: Pass the final hidden state through the fully connected layer
        logits = self.fc(final_hidden)
        return logits

model = SimpleLSTM(vocab_size=len(itos), embed_dim=64, hidden_dim=128, num_classes=4).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 4,260,932


### Step 3: Train and Evaluate
Write the core steps of the PyTorch training loop.

In [3]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for texts, labels in tr_ld:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        # TODO: Perform a forward pass
        predictions = model(texts)

        # TODO: Compute the loss
        loss = criterion(predictions, labels)

        # TODO: Perform backpropagation
        loss.backward()

        # TODO: Update the weights
        optimizer.step()

        total_loss += loss.item()
        correct += (predictions.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Evaluation Phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for texts, labels in te_ld:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts)
            val_correct += (preds.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/3 | Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")

Epoch 1/3 | Train Acc: 29.4% | Val Acc: 25.5%
Epoch 2/3 | Train Acc: 30.4% | Val Acc: 25.7%
Epoch 3/3 | Train Acc: 30.3% | Val Acc: 25.8%


### Step 4: Reflection
1. What does the `padding_idx` argument do in the `nn.Embedding` layer?
2. Why do we extract `hidden[-1]` instead of using the raw `output` from the LSTM for classification?

### Step 5: Inference
Now that the model is trained, let's use it to classify a brand new headline that it has never seen before.

In [4]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text, model):
    model.eval()

    # TODO: Convert the raw text into a tensor of token ids using numericalize().
    # Hint: wrap the result in torch.tensor(..., dtype=torch.int64), then move it to `device`.
    text_tensor = torch.tensor(numericalize(text), dtype=torch.int64).to(device)

    # TODO: The model expects a batch dimension. Add one with .unsqueeze(0).
    text_tensor = text_tensor.unsqueeze(0)

    with torch.no_grad():
        # TODO: Run a forward pass through the model to get the logits.
        logits = model(text_tensor)

        # TODO: Get the predicted class index from the logits (highest score).
        # Hint: use .argmax(1) and .item()
        predicted_idx = logits.argmax(1).item()

    return class_names[predicted_idx]

sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Central bank raises interest rates to combat inflation",
    "NASA's new telescope captures images of distant galaxy",
    "Peace talks resume between the two neighboring countries"
]

for headline in sample_headlines:
    prediction = predict(headline, model)
    print(f"'{headline}' -> {prediction}")

'Manchester United wins dramatic final in extra time' -> Sci/Tech
'Central bank raises interest rates to combat inflation' -> Sports
'NASA's new telescope captures images of distant galaxy' -> World
'Peace talks resume between the two neighboring countries' -> Sci/Tech
